In [1]:
import pandas as pd

In [3]:
benchmark = pd.read_json("../examples/acknowledgments/benchmark_acknowledgments.json")
benchmark.head(5)

,text,publication_id,funders,projects,infrastructure,persons_thanked,private_companies,persons
0,Acknowledgments This research is part of a col...,doi10.17645/pag.v7i3.2122,"[{'mention': 'ANR', 'canonical_name': 'Agence ...","[{'mention': 'CLAIMS', 'name': 'CLAIMS', 'fund...",[],"[{'name': 'Brigitte Geissel', 'role': 'insight...",[],NaN
1,This work was supported by the National Natura...,doi10.1101/2022.02.26.482011,[{'mention': 'National Natural Science Foundat...,[{'mention': 'National Natural Science Foundat...,[],[],[],NaN
2,The research was supported in part by the DAAD...,doi10.17645/pag.v7i3.2122,[{'mention': 'Bundesministeriums für Bildung u...,"[{'mention': 'PPP USA 2018, 57387214', 'grant_...",[],[],[],NaN
3,Acknowledgments This research was supported by...,doi10.37236/11050,"[{'mention': 'ANR', 'canonical_name': 'Agence ...",[{'mention': 'ANR project DAGDigDec (JCJC ) ...,[],[],[],NaN
4,Je remercie également le projet GEM Flagship f...,hallirmm-03326344,"[{'mention': 'ANR', 'canonical_name': 'Agence ...",[{'mention': 'projet GEM Flagship financé par ...,[],[],[],NaN


In [2]:
outputs = pd.read_json("../examples/acknowledgments/benchmark_output.json")
outputs

,publication_id,text,llm_output
0,doi10.17645/pag.v7i3.2122,Acknowledgments This research is part of a col...,\n### 1. Initial scan\n\nScientific acknowledg...
1,doi10.1101/2022.02.26.482011,This work was supported by the National Natura...,\n**Initial scan:** Clean funding statement. E...
2,doi10.17645/pag.v7i3.2122,The research was supported in part by the DAAD...,\nScanning text. Formal funding acknowledgemen...
3,doi10.37236/11050,Acknowledgments This research was supported by...,\n**Initial scan**: Clean English acknowledgme...
4,hallirmm-03326344,Je remercie également le projet GEM Flagship f...,"\n### 1. Initial scan\n\nFrench text. ""Je reme..."
5,doi10.1109/infocom.2019.8737600,*The work was done when the author was with H...,"\n**Initial scan.** Clean English text, formal..."
6,doi10.1016/j.cam.2018.06.027,Acknowledgments.This work has been partially f...,\n### 1. Initial scan\n\nClean acknowledgment ...
7,doi10.48550/arxiv.2405.10361,We thank the referee for the detailed comments...,\n### 1. Initial scan\n\nAstronomy acknowledge...
8,doi10.1590/s1517-707620180002.0375,AGRADECIMIENTOS Al CODI de la Universidad de A...,"\nSpanish text, formal register. ""AGRADECIMIEN..."
9,doi10.37044/osf.io/9mnkb,Acknowledgements This work was done during the...,\n### 1. Initial scan\n\nBioHackathon Europe 2...


In [6]:
test = outputs[outputs.publication_id == "doi10.1063/1.4807558"]["llm_output"].values[0]
len(test) / 4

1336.0

In [24]:
outputs["parsed_output"] = outputs["llm_output"].apply(
    lambda x: "{" + x.split("\n\n{")[1]
)
outputs["parsed_output"] = outputs["parsed_output"].apply(
    lambda x: x.replace('infrastructure":', 'infrastructures":')
)
outputs["parsed_output"].tail(5)

16    {"funders": [{"mention": "GENCI", "canonical_n...
17    {"funders": [{"mention": "ANR", "canonical_nam...
18    {"funders": [{"mention": "Agence Nationale de ...
19    {"funders": [{"mention": "Deutsche Forschungsg...
20    {"funders": [{"mention": "European Union", "ca...
Name: parsed_output, dtype: object

In [22]:
benchmark["expectations"] = benchmark.apply(
    lambda row: {
        "funders": row["funders"],
        "projects": row["projects"],
        "infrastructures": row["infrastructure"],
        "persons_thanked": row["persons_thanked"],
        "private_companies": row["private_companies"],
        "persons": row["persons"],
    },
    axis=1,
)
benchmark["expectations"].head(5)

0    {'funders': [{'mention': 'ANR', 'canonical_nam...
1    {'funders': [{'mention': 'National Natural Sci...
2    {'funders': [{'mention': 'Bundesministeriums f...
3    {'funders': [{'mention': 'ANR', 'canonical_nam...
4    {'funders': [{'mention': 'ANR', 'canonical_nam...
Name: expectations, dtype: object

In [34]:
evaluation = pd.merge(
    benchmark[["publication_id", "text", "expectations"]],
    outputs[["publication_id", "parsed_output"]],
    on="publication_id",
)
evaluation.head(5)

,publication_id,text,expectations,parsed_output
0,doi10.17645/pag.v7i3.2122,Acknowledgments This research is part of a col...,"{'funders': [{'mention': 'ANR', 'canonical_nam...","{""funders"": [{""mention"": ""ANR"", ""canonical_nam..."
1,doi10.17645/pag.v7i3.2122,Acknowledgments This research is part of a col...,"{'funders': [{'mention': 'ANR', 'canonical_nam...","{""funders"": [{""mention"": ""Bundesministeriums f..."
2,doi10.1101/2022.02.26.482011,This work was supported by the National Natura...,{'funders': [{'mention': 'National Natural Sci...,"{""funders"": [{""mention"": ""National Natural Sci..."
3,doi10.17645/pag.v7i3.2122,The research was supported in part by the DAAD...,{'funders': [{'mention': 'Bundesministeriums f...,"{""funders"": [{""mention"": ""ANR"", ""canonical_nam..."
4,doi10.17645/pag.v7i3.2122,The research was supported in part by the DAAD...,{'funders': [{'mention': 'Bundesministeriums f...,"{""funders"": [{""mention"": ""Bundesministeriums f..."


In [47]:
import json

evaluation["expectations"] = evaluation["expectations"].apply(lambda x: json.dumps(x))

In [51]:
evaluation.to_json(
    "../examples/acknowledgments/benchmark_evaluation.json", orient="records", indent=2
)

In [32]:
evaluation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   publication_id  23 non-null     object
 1   text            23 non-null     object
 2   expectations    23 non-null     object
 3   parsed_output   23 non-null     object
dtypes: object(4)
memory usage: 868.0+ bytes


In [33]:
evaluation.iloc[0]["text"]

'Acknowledgments This research is part of a collective research project jointly funded by  ANR  and  DFG , called CLAIMS (https://claims.hypotheses.org/68);it greatly bene-fited from discussions within the team.I also want to thank the two anonymous reviewers as well as  Brigitte Geissel  and  Petra Guasti  for their insightful comments and suggestions.'

In [37]:
e = evaluation.iloc[0]["expectations"]
print(type(e))
print(type(e.get("funders")))
print(type(e.get("funders")[0].get("mention")))

<class 'dict'>
<class 'list'>
<class 'str'>


In [54]:
from datasets import load_dataset

file_path = "../examples/acknowledgments/benchmark_evaluation.json"
dataset = load_dataset("json", data_files={"file": file_path}, split="file")
df = dataset.to_pandas()
df.info()

Generating file split: 0 examples [00:00, ? examples/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   publication_id  23 non-null     object
 1   text            23 non-null     object
 2   expectations    23 non-null     object
 3   parsed_output   23 non-null     object
dtypes: object(4)
memory usage: 868.0+ bytes


In [57]:
df[df["publication_id"] == "doi10.1063/1.4807558"].parsed_output.values[0]

'{"funders": [{"mention": "United States National Science Foundation", "canonical_name": "National Science Foundation", "funder_short": "NSF", "country": "US", "grant_ids": [], "programs": []}, {"mention": "Science and Technology Facilities Council of the United Kingdom", "canonical_name": "Science and Technology Facilities Council", "funder_short": "STFC", "country": "UK", "grant_ids": [], "programs": [], "wikidata_qid": "Q7394670", "reconciled_name": "Science and Technology Facilities Council"}, {"mention": "Max-Planck-Society", "canonical_name": "Max-Planck-Gesellschaft", "funder_short": "MPG", "country": "DE", "grant_ids": [], "programs": []}, {"mention": "State of Niedersachsen/Germany", "canonical_name": "Land Niedersachsen", "funder_short": null, "country": "DE", "grant_ids": [], "programs": [], "wikidata_qid": "Q17219540", "reconciled_name": "Land Niedersachsen"}, {"mention": "Istituto Nazionale di Fisica Nucleare of Italy", "canonical_name": "Istituto Nazionale di Fisica Nucle

In [58]:
outputs[outputs.publication_id == "doi10.1063/1.4807558"].llm_output.values[0]

'\n**Initial scan.** Scientific acknowledgement, formal register. "LIGO Laboratory", "GEO600 detector", "Virgo detector"  gravitational wave physics. International collaboration pattern emerging.\n\n**Discipline context.** LIGO/Virgo = major gravitational wave detectors. High-energy physics/astrophysics. Expect: major international funding agencies, consortium structure.\n\n**Entity discovery sequence:**\n\n**United States National Science Foundation**  NSF  Standard US federal funder. Supports LIGO construction/operation.\n\n**Science and Technology Facilities Council of the United Kingdom**  STFC  Q7394670. UK research council, physics/astronomy focus. Makes sense for gravitational wave detector.\n\n**Max-Planck-Society**  Max-Planck-Gesellschaft  German research organization. "Max-Planck-Gesellschaft" is canonical German name.\n\n**State of Niedersachsen/Germany**  Land Niedersachsen  Q17219540. German federal state. "Niedersachsen" = Lower Saxony. Regional funding for German detect

In [52]:
type(df.iloc[0].expectations.get("funders"))

AttributeError: 'str' object has no attribute 'get'